# 🎙️ HF Speech-to-Speech on a free Colab GPU

**Architecture:** all heavy models (Parakeet STT, Qwen3-4B LLM, Qwen3-TTS) run on Colab's **free T4 GPU**.
Your Mac runs only the lightweight mic/speaker client (`talk`) — no models, no heat, no slowdown.

```
Your Mac (mic + speaker)  ⇄  Cloudflare tunnel  ⇄  Colab T4 (full pipeline)
        talk client                      wss://                serve on :8765
```

**Steps:**
1. In Colab: **Runtime → Change runtime type → T4 GPU → Save**
2. Run the cells top to bottom
3. Copy the tunnel URL from Step 3 into the `talk` command on your Mac (Step 4)

In [ ]:
# Step 0 — confirm we actually have a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Step 1 — system audio libs + the package
# Colab is Ubuntu 22.04 (glibc 2.35), but the default Qwen3-TTS wheel targets
# CUDA 12.8 + glibc 2.39, so we pre-install the matching wheel (per the repo README).
# If the cu124 line errors, comment it out and use the cpu line instead.
!apt-get update -qq && apt-get install -y -qq libportaudio2 libsndfile1 > /dev/null 2>&1
!pip install -q "qwentts-cpp-python==0.3.1+cu124" -f https://huggingface.co/datasets/andito/qwentts-cpp-python-wheels/tree/main/whl/cu124
# !pip install -q "qwentts-cpp-python==0.3.1+cpu" -f https://huggingface.co/datasets/andito/qwentts-cpp-python-wheels/tree/main/whl/cpu  # fallback
!pip install -q speech-to-speech
!which speech-to-speech && echo "install OK"

## Step 2 — start the realtime server (background)

**Recommended: hosted LLM (fast start).** Loading the 4B LLM's weights from Colab's slow disk
takes hours (398 shards, one disk-read + GPU-copy each). Instead, STT + TTS run on the Colab GPU
while the LLM runs via Hugging Face's inference API (free credits) — the server is up in ~2–3 minutes,
and swapping the LLM later is just changing `--model_name`.

Get a free token first: huggingface.co → Settings → Access Tokens → create a **read** token.


In [ ]:
# HF token: read from Colab Secrets (key icon in left sidebar -> save as HF_TOKEN
# with 'Notebook access' on). Falls back to asking once if not saved.
import os
_tok = None
try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
except Exception:
    _tok = None
if not _tok:
    from getpass import getpass
    _tok = getpass("Paste your HF read token (or save it in Colab Secrets as HF_TOKEN): ")
os.environ["HF_TOKEN"] = _tok

# Qwen3-TTS loads libcudart via ctypes; pip-installed CUDA libs are not on the
# system loader path, so expose them (must run before the server starts).
import sysconfig, glob
_purelib = sysconfig.get_path("purelib")
_nv_libs = sorted({d for d in glob.glob(os.path.join(_purelib, "nvidia", "*", "lib")) if os.path.isdir(d)})
if _nv_libs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(_nv_libs) + (":" + os.environ["LD_LIBRARY_PATH"] if os.environ.get("LD_LIBRARY_PATH") else "")
    print("LD_LIBRARY_PATH set,", len(_nv_libs), "nvidia lib dirs")
else:
    print("WARNING: no nvidia CUDA libs found")

os.makedirs("/content/s2s", exist_ok=True)

# stop any previous server (bracket trick keeps pkill from matching its own shell)
import subprocess, time
subprocess.run(["pkill", "-f", "[s]peech-to-speech serve"])
time.sleep(2)

# Launch via subprocess (NOT `! ... &`): a backgrounded `!` holds the cell's output
# pipe open, hanging the cell forever once the server stays alive.
# To try the fastest LLM route (Groq), swap the --model_name value for "openai/gpt-oss-20b:groq".
_args = ["speech-to-speech", "serve",
         "--host", "0.0.0.0",
         "--device", "cuda",
         "--stt", "parakeet-tdt",
         "--llm_backend", "responses-api",
         "--model_name", "Qwen/Qwen3.5-9B:together",
         "--responses_api_base_url", "https://router.huggingface.co/v1",
         "--responses_api_api_key", os.environ["HF_TOKEN"],
         "--responses_api_stream",
         "--responses_api_reasoning_effort", "low",
         "--tts", "qwen3",
         "--qwen3_tts_backend", "ggml"]
_log = open("/content/s2s/server.log", "ab")
subprocess.Popen(_args, stdout=_log, stderr=subprocess.STDOUT,
                 stdin=subprocess.DEVNULL, start_new_session=True, cwd="/content/s2s")
print("server launched in background — check the log in the next cell")


In [ ]:
# Watch warmup progress (re-run this cell to refresh).
# You're ready when the log shows the server listening on :8765.
!tail -n 40 /content/s2s/server.log

## Step 3 — expose the server (Colab blocks inbound connections, so we tunnel out)

Cloudflare quick tunnel: no account needed. The URL changes every run — copy the fresh one each time.

In [ ]:
# Stable speech tunnel + tool-API tunnel.
# The Mac `talk` command keeps using the stable URL (never changes).
# Tool EXECUTION now runs on Colab: a tiny API server on port 8766,
# exposed via a second ngrok tunnel. That URL changes per session —
# paste it once as TOOL_API_URL in the Mac terminal.
# One-time setup (~5 min):
#   1. Sign up at https://ngrok.com (free) -> copy your authtoken
#   2. Dashboard -> Domains -> claim a free static domain, e.g. akhil-s2s.ngrok-free.app
#   3. Colab Secrets (key icon, 'Notebook access' on): NGROK_TOKEN=<authtoken>,
#      NGROK_DOMAIN=<your static domain>
import os, re, subprocess, time, shutil
try:
    from google.colab import userdata
    _ngrok_token = userdata.get('NGROK_TOKEN')
    _ngrok_domain = userdata.get('NGROK_DOMAIN')
except Exception:
    _ngrok_token, _ngrok_domain = None, None
if not _ngrok_token:
    from getpass import getpass
    _ngrok_token = getpass('ngrok authtoken: ')
    _ngrok_domain = input('ngrok static domain (e.g. akhil-s2s.ngrok-free.app): ').strip()

if not shutil.which('ngrok'):
    print('installing ngrok...')
    import urllib.request, tarfile
    urllib.request.urlretrieve('https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz', '/tmp/ngrok.tgz')
    with tarfile.open('/tmp/ngrok.tgz') as t:
        t.extract('ngrok', '/tmp', filter='data')
    shutil.move('/tmp/ngrok', '/usr/local/bin/ngrok')
    os.chmod('/usr/local/bin/ngrok', 0o755)
    print('ngrok installed')

# kill previous tunnels/servers so re-runs are clean
subprocess.run(['pkill', '-f', '[n]grok http 8765'], capture_output=True)
subprocess.run(['pkill', '-f', '[n]grok http 8766'], capture_output=True)
subprocess.run(['pkill', '-f', '[t]ool_server.py'], capture_output=True)
subprocess.run(['pkill', '-f', '[c]loudflared tunnel'], capture_output=True)
time.sleep(1)
subprocess.run(['ngrok', 'config', 'add-authtoken', _ngrok_token], capture_output=True)

# ---- 1. main speech tunnel (stable URL) ----
_log = open('/content/s2s/ngrok.log', 'ab')
subprocess.Popen(['ngrok', 'http', '8765', f'--url=https://{_ngrok_domain}'],
                 stdout=_log, stderr=subprocess.STDOUT,
                 stdin=subprocess.DEVNULL, start_new_session=True)

# ---- 2. tool API server: the REAL tool logic runs here on Colab ----
_tool_server_code = '''
import json
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

_jobs = {}

def _run_backend_job(search_id, claim_number):
    time.sleep(10)
    _jobs[search_id] = {
        "done": True,
        "result": {
            "claim_number": claim_number,
            "status": "In review",
            "eta_days": 3,
            "last_update": "Assigned to adjuster Priya Nair",
        },
    }

class Handler(BaseHTTPRequestHandler):
    def _send(self, obj):
        body = json.dumps(obj).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_POST(self):
        if self.path != "/run-tool":
            self.send_error(404)
            return
        length = int(self.headers.get("Content-Length", 0))
        try:
            req = json.loads(self.rfile.read(length) or b"{}")
        except Exception:
            req = {}
        name = req.get("name")
        args = req.get("arguments", {}) or {}
        if name == "lookup_claim_status":
            search_id = "SRCH-" + uuid.uuid4().hex[:6]
            _jobs[search_id] = {"done": False}
            threading.Thread(
                target=_run_backend_job,
                args=(search_id, args.get("claim_number", "unknown")),
                daemon=True,
            ).start()
            self._send({"search_id": search_id, "status": "pending", "eta_seconds": 10})
        elif name == "get_claim_result":
            job = _jobs.get(args.get("search_id"))
            if not job:
                self._send({"status": "error", "note": "unknown search_id"})
            elif not job["done"]:
                self._send({"status": "still_pending",
                            "note": "Tell the user it is still running; offer to check again shortly."})
            else:
                self._send({"status": "complete", **job["result"]})
        else:
            self._send({"status": "error", "note": "unknown tool: " + str(name)})

    def log_message(self, *args):
        pass

print("tool server starting on 127.0.0.1:8766", flush=True)
ThreadingHTTPServer(("127.0.0.1", 8766), Handler).serve_forever()
'''
open('/content/s2s/tool_server.py', 'w').write(_tool_server_code)
_tls = open('/content/s2s/tool_server.log', 'ab')
subprocess.Popen(['python3', '/content/s2s/tool_server.py'],
                 stdout=_tls, stderr=subprocess.STDOUT,
                 stdin=subprocess.DEVNULL, start_new_session=True)

# ---- 3. second ngrok tunnel for the tool API (ephemeral URL) ----
_tlog = open('/content/s2s/ngrok-tools.log', 'ab')
subprocess.Popen(['ngrok', 'http', '8766'],
                 stdout=_tlog, stderr=subprocess.STDOUT,
                 stdin=subprocess.DEVNULL, start_new_session=True)
time.sleep(9)
_tool_url = None
_blog = open('/content/s2s/ngrok-tools.log').read()
_m = re.search(r'Forwarding\s+(https://\S+)\s+->', _blog)
if _m:
    _tool_url = _m.group(1)

print('YOUR STABLE URL: https://' + _ngrok_domain)
print('Mac talk url:    wss://' + _ngrok_domain + '/v1/realtime')
print()
if _tool_url:
    print('TOOL_API_URL (export on Mac, once per Colab session):')
    print('export TOOL_API_URL=' + _tool_url)
else:
    print('WARNING: tools-tunnel URL not found — check /content/s2s/ngrok-tools.log')


## Step 4 — on your Mac: the lightweight client

This installs **only the mic/speaker client** — no models are downloaded, so your Air stays fast.

```bash
python3 -m venv s2s-client
source s2s-client/bin/activate
pip install speech-to-speech
export OPENAI_API_KEY=not-needed  # dummy value — the server ignores it, but the SDK requires one
speech-to-speech talk --url wss://<paste-your-tunnel-url-here>/v1/realtime --playback-buffer-ms 800 --block-mic-during-playback
```

Allow mic access when prompted, use headphones, speak and pause for the reply. `Ctrl+C` to stop.

## Tips

- **Per-stage latency is logged on the server per response** (STT, LLM, first-TTS-audio, speech-to-audio durations) — `!tail -f /content/s2s/server.log` is your latency lab.
- Colab kills runtimes after ~12h (sooner if idle). Re-run the cells to restart; models live only in the VM, so a fresh VM re-downloads them (~10–20 min). Keep the tab open during experiments.
- Stop the server: `!pkill -f speech-to-speech`
- With the ngrok static domain the URL never changes — set the Mac `talk` command once and reuse it.
If the voice stutters or cuts off mid-sentence, raise `--playback-buffer-ms` (e.g. 1200) — it trades a slightly later start for smooth playback over the network tunnel.

`--block-mic-during-playback` pauses the mic while the assistant speaks, so it can't hear itself through your speakers (no barge-in while this is on — use headphones instead if you want to interrupt it).
